# 6장 2강: 허깅페이스 모델 실습
## 2. 트랜스포머 아키텍처별 모델 실습


### 2.2 인코더 모델 활용: KoBERT로 감정 분류

In [6]:
# 토크나이저 및 모델 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# KoBERT 토크나이저와 모델 로드
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("rkdaldus/ko-sent5-classification")

# 사용자 입력 텍스트 감정 분석
#text = "오늘 정말 행복해!"
text = "너무 무서워!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
print("inputs:", inputs)
with torch.no_grad():
    outputs = model(**inputs)
predicted_label = torch.argmax(outputs.logits, dim=1).item()

# 감정 레이블 정의
emotion_labels = {
    0: ("Angry", "😡"),
    1: ("Fear", "😨"),
    2: ("Happy", "😊"),
    3: ("Tender", "🥰"),
    4: ("Sad", "😢")
}

# 예측된 감정 출력
print(f"예측된 감정: {emotion_labels[predicted_label][0]} {emotion_labels[predicted_label][1]}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3573.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rkdaldus/ko-sent5-classification
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


inputs: {'input_ids': tensor([[   2, 1458, 2095, 6553, 7018,    5,    3]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}
예측된 감정: Sad 😢


### 2.3 인코더-디코더 모델 활용: KoBART로 뉴스 요약

In [8]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

#text = "과거를 떠올려보자. 방송을 보던 우리의 모습을. 독보적인 매체는 TV였다. 온 가족이 둘러앉아 TV를 봤다. 간혹 가족들끼리 뉴스와 드라마, 예능 프로그램을 둘러싸고 리모컨 쟁탈전이 벌어지기도  했다. 각자 선호하는 프로그램을 ‘본방’으로 보기 위한 싸움이었다. TV가 한 대인지 두 대인지 여부도 그래서 중요했다. 지금은 어떤가. ‘안방극장’이라는 말은 옛말이 됐다. TV가 없는 집도 많다. 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다. 각자의 방에서 각자의 휴대폰으로, 노트북으로, 태블릿으로 콘텐츠 를 즐긴다."
text = """쿡은 애플에서 대표적인 ‘운영 전문가’였다. 1998년 애플에 합류한 뒤 최고운영책임자(COO)로 글로벌 영업과 공급망을 총괄했고, 2011년 스티브 잡스의 뒤를 이어 CEO에 올랐다. 팀 쿡 CEO는 ‘혁신의 시대’를 열었던 스티브 잡스를 이어 애플을 책임지며 상대적으로 박한 평가를 받았다. 잡스가 마치 발명가처럼 아이팟(2001년)·아이폰(2007년) 등 세상에 없던 혁신 제품을 내놓으며 시장에 충격을 줬지만, 쿡 CEO는 그렇지 못했기 때문이다. 하지만 쿡 CEO는 잡스가 쌓아 올린 애플이라는 성(城)의 울타리를 높이고, 내부 생태계를 탄탄히 만들었다는 평가를 받는다. 혁신 대신 수익성을 높인 ‘확장의 시대’를 연 것이다."""

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]))
tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)


Loading weights: 100%|██████████| 260/260 [00:00<00:00, 2276.65it/s]
c:\Users\yonggyo\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\transformers\generation\utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


"팀 쿡 CEO는 혁신 대신 수익성을 높인 '확장의 시대'를 열었으며, 애플을"

### 2.4 디코더 모델 활용: Gemma로 대화형 텍스트 생성

In [1]:
from transformers import pipeline
import torch

gemma_identifier = "google/gemma-2b-it"

gemma_generator = pipeline(
    "text-generation",
    model=gemma_identifier,
    dtype=torch.bfloat16,
    device_map="auto" # accelerate
)

"""
role: system : 역할, 컨텍스트...
      user: 전달할 대화
      assistant: AI가 답변한 내용, 지난 대화 내용 +(user + assitant)
"""
user_dialogue = [
    {"role": "user", "content": "내 이름은 무엇이지?"}
]

outputs = gemma_generator(user_dialogue, max_new_tokens=150)
outputs

c:\Users\yonggyo\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yonggyo\Documents\ax_study\04_MACHINE_LEARNING\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yonggyo\.cache\huggingface\hub\models--google--gemma-2b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Py

[{'generated_text': [{'role': 'user', 'content': '내 이름은 무엇이지?'},
   {'role': 'assistant',
    'content': '저는 현재 이름이 없습니다. 저는 예술적으로 움직이는 AI로, 저는 계속적으로 변화하고 발전하는 과정을 거치고 있습니다.'}]}]

In [2]:
user_dialogue = [
    {"role": "user", "content": "내이름은 김철수야"},
    {"role": "assistant", "content": "반갑습니다. 철수님!"},
    {"role": "user", "content": "내 이름은 무엇이지?"}
]

outputs = gemma_generator(user_dialogue, max_new_tokens=150)
outputs

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': '내이름은 김철수야'},
   {'role': 'assistant', 'content': '반갑습니다. 철수님!'},
   {'role': 'user', 'content': '내 이름은 무엇이지?'},
   {'role': 'assistant', 'content': '김철수입니다. 감사합니다!\n\n저는 김철수라고 불리는 사람입니다.'}]}]